# Lab 3：RAG + Grader — 過濾不相關文件

## 學習目標
1. 建立簡單的 **RAG**（檢索 + 生成）
2. 加入 **Grader**：讓 LLM 判斷文件是否相關，過濾後再生成
3. 比較有無 Grader 的回答差異

## 流程
```
問題 → 檢索文件 → [Grader 過濾] → 只留相關文件 → LLM 生成回答
```

In [ ]:
%%capture
!pip install langchain-openai langchain-core chromadb

In [ ]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage
import chromadb

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 建立 HP 知識庫
documents = [
    "HP EliteBook 855 G10 搭載 AMD Ryzen 7 PRO 7840U，32GB DDR5，512GB NVMe SSD，支援 WiFi 6E。",
    "HP EliteBook 855 G10 電池容量 51Wh，一般使用續航約 10-12 小時，支援快充。",
    "HP ProBook 450 G10 搭載 Intel Core i7-1355U，16GB DDR4，256GB NVMe SSD。",
    "HP 企業筆電提供 3 年保固，可選購延長保固及意外損壞保護。",
    "HP 筆電通過 MIL-STD-810H 軍規認證，耐高低溫、震動與跌落測試。",
]

chroma = chromadb.Client()
collection = chroma.create_collection("hp_docs")
doc_embeddings = embeddings.embed_documents(documents)
collection.add(
    ids=[str(i) for i in range(len(documents))],
    embeddings=doc_embeddings,
    documents=documents
)

def retrieve(query: str, top_k: int = 3) -> list[str]:
    q_emb = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    return results["documents"][0]

print(f"✓ 知識庫建立完成（{len(documents)} 份文件）")

## Step 1：Naive RAG — 不過濾，直接生成

先跑一次 Naive RAG，看看不做過濾時的效果。

In [ ]:
def generate(query: str, docs: list[str]) -> str:
    context = "\n".join(docs)
    return llm.invoke([
        SystemMessage(content="你是 HP 客服，只根據提供的資料回答，資料不足時請說不知道。"),
        HumanMessage(content=f"資料：\n{context}\n\n問題：{query}")
    ]).content

# Naive RAG
query = "HP EliteBook 855 的電池可以用多久？"
docs = retrieve(query)
print(f"檢索到 {len(docs)} 份文件：")
for i, d in enumerate(docs): print(f"  [{i+1}] {d[:60]}...")
print(f"\n回答：{generate(query, docs)}") 

## Step 2：加入 Grader — 讓 LLM 判斷文件相關性

Grader 的工作：對每份文件問 LLM「這份文件跟問題有關嗎？」  
只保留相關的文件，減少 LLM 被無關資訊干擾的機會。

In [ ]:
# =============================================
# TODO：完成 Grader 函數（約 6 行）
#
# 提示：
#   1. 用 llm.invoke 請 LLM 判斷文件是否與問題相關
#   2. 要求 LLM 只回答 "yes" 或 "no"
#   3. 解析回應，回傳 True / False
# =============================================
def grade_document(query: str, document: str) -> bool:
    """判斷文件是否與問題相關，回傳 True/False"""
    pass  # 請實作這裡


# 測試 Grader
print("Grader 測試：")
print(f"  相關文件：{grade_document(query, docs[0])}")  # 應為 True
print(f"  不相關文件：{grade_document(query, '台北天氣晴朗')}")  # 應為 False

## Step 3：比較 Naive RAG vs Graded RAG

In [ ]:
def graded_rag(query: str) -> str:
    docs = retrieve(query, top_k=5)
    relevant = [d for d in docs if grade_document(query, d)]
    print(f"  檢索 {len(docs)} 份 → 過濾後剩 {len(relevant)} 份")
    if not relevant:
        return "找不到相關資料。"
    return generate(query, relevant)

test_queries = [
    "EliteBook 855 的記憶體有多大？",   # 有相關文件
    "HP 筆電怎麼重灌 Windows？",        # 知識庫沒有這個資訊
]

for q in test_queries:
    print("=" * 50)
    print(f"問：{q}")
    naive = generate(q, retrieve(q))
    print(f"Naive RAG：{naive[:100]}")
    graded = graded_rag(q)
    print(f"Graded RAG：{graded[:100]}")